# Nuvii LoRA v2 — balanced vocabulary + 768 px experiment

Use this **after** improving caption balance. It is not a replacement for the working 512 px model until the new dataset passes evaluation.

- Base model: `segmind/SSD-1B`
- Default resolution: 768 px
- LoRA rank: 8
- Batch size: 1
- Gradient checkpointing + 8-bit Adam
- Checkpoints every 200 steps

If a T4 runs out of VRAM at 768, change `RESOLUTION = 640` and rerun.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, zipfile

INPUT_ROOT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
TRAIN_DIR = WORK / 'nuvii_v2_train'
OUTPUT_DIR = WORK / 'nuvii_v2_lora'
CACHE_DIR = WORK / 'hf_cache_v2'
RESOLUTION = 768
MAX_STEPS = 1600

subprocess.run(['nvidia-smi'], check=False)
subprocess.run(['free','-h'], check=False)
print('Resolution:', RESOLUTION, 'Steps:', MAX_STEPS)


In [ ]:
# Find a train_imagefolder supplied directly or inside a ZIP.
if TRAIN_DIR.exists(): shutil.rmtree(TRAIN_DIR)
TRAIN_DIR.mkdir(parents=True)

folders = [p for p in INPUT_ROOT.rglob('train_imagefolder') if p.is_dir()]
if folders:
    for item in folders[0].iterdir():
        if item.is_file(): shutil.copy2(item, TRAIN_DIR / item.name)
else:
    zips = [p for p in INPUT_ROOT.rglob('*.zip') if 'nuvii' in p.name.lower()]
    if not zips: raise FileNotFoundError('Attach the balanced Nuvii dataset first.')
    with zipfile.ZipFile(zips[0]) as zf:
        members=[n for n in zf.namelist() if '/train_imagefolder/' in n and not n.endswith('/')]
        for member in members:
            with zf.open(member) as src, open(TRAIN_DIR/Path(member).name,'wb') as dst:
                shutil.copyfileobj(src,dst)

images=[p for p in TRAIN_DIR.iterdir() if p.suffix.lower() in {'.png','.jpg','.jpeg','.webp'}]
captions=list(TRAIN_DIR.glob('*.txt'))
print('Images:',len(images),'Captions:',len(captions))
if not images or len(images)!=len(captions): raise RuntimeError('Images/captions must be paired 1:1.')


In [ ]:
# Quick concept-coverage audit before spending GPU time.
CONCEPTS = {
 'french_tip':['french tip','french'], 'aura':['aura'], 'ombre':['ombre','gradient'],
 'bow':['bow','ribbon'], 'rhinestone':['rhinestone','crystal','gem'], 'pearl':['pearl'],
 'chrome':['chrome','metallic'], '3d_flower':['3d flower','raised flower','orchid'],
 'swirl':['swirl','ripple'], 'shell':['shell'], 'bubble':['bubble','raindrop'],
 'caviar_beads':['caviar','micro bead'], 'stars':['star','starburst','sparkle'],
 'animal_print':['cheetah','leopard','zebra'], 'butterfly':['butterfly']
}
texts=[p.read_text(errors='ignore').lower() for p in captions]
for name,terms in CONCEPTS.items():
    count=sum(any(t in text for t in terms) for text in texts)
    print(f'{name:16} {count:3}', 'OK' if count>=20 else '<-- add examples')


In [ ]:
subprocess.run([
 sys.executable,'-m','pip','install','-q',
 'git+https://github.com/huggingface/diffusers.git',
 'transformers','accelerate','datasets','peft','bitsandbytes','xformers','safetensors','sentencepiece','tensorboard'
], check=True)
# Kaggle sometimes preinstalls an incompatible torchao; the LoRA path does not need it.
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'], check=False)
print('Libraries ready.')


In [ ]:
import urllib.request
SCRIPT = WORK/'train_text_to_image_lora_sdxl.py'
urllib.request.urlretrieve(
 'https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/train_text_to_image_lora_sdxl.py',
 SCRIPT
)
print(SCRIPT)


In [ ]:
# Train. Keep the notebook attached until the final ZIP is downloaded.
if OUTPUT_DIR.exists(): shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

cmd=[
 sys.executable,'-m','accelerate.commands.launch','--num_processes=1','--mixed_precision=fp16',str(SCRIPT),
 '--pretrained_model_name_or_path=segmind/SSD-1B','--variant=fp16',
 f'--train_data_dir={TRAIN_DIR}',f'--output_dir={OUTPUT_DIR}',f'--cache_dir={CACHE_DIR}',
 f'--resolution={RESOLUTION}','--center_crop','--train_batch_size=1','--gradient_accumulation_steps=4',
 '--gradient_checkpointing',f'--max_train_steps={MAX_STEPS}','--learning_rate=1e-4',
 '--lr_scheduler=cosine','--lr_warmup_steps=100','--snr_gamma=5.0','--checkpointing_steps=200',
 '--checkpoints_total_limit=4','--mixed_precision=fp16','--enable_xformers_memory_efficient_attention',
 '--use_8bit_adam','--dataloader_num_workers=0','--rank=8','--seed=42','--report_to=tensorboard'
]
env=os.environ.copy(); env['CUDA_VISIBLE_DEVICES']='0'; env['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
subprocess.run(cmd, env=env, check=True)
print('TRAINING FINISHED')


In [ ]:
# Back up immediately.
archive=shutil.make_archive('/kaggle/working/nuvii_v2_lora_backup','zip',root_dir=OUTPUT_DIR)
print('DOWNLOAD:',archive)
